In [ ]:
# Imports
import torch
from torch import nn

from random import randint

# Generating the Data

In [ ]:
# Generating data
X = [(randint(0, 10), randint(0, 10)) for i in range(20)]
y = [n1*n2 for (n1, n2) in X]

# Transforming data and converting dtypes to float32 as model weights 
# and predictions are of this type. Also normalizes data and 
# unsqueezes to get correct shape.
X = torch.tensor(X, dtype=torch.float32) / 10
y = torch.tensor(y, dtype=torch.float32).unsqueeze(1) / 100

# Model Setup

In [ ]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self, width):
        super().__init__()

        # We use ReLU to introduce nonlinearity
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(2, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, 1)
        )
    
    def forward(self, x):
        logits = self.linear_relu_stack(x) # Logits are the outputs
        return logits

In [ ]:
# Method used for training
def train(X, y, model, loss_fn, optimizer, epochs=20, device='cpu'):
    model.train()
    model, X, y = model.to(device), X.to(device), y.to(device)
    for epoch in range(epochs+1):

        # Forward pass
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad() # Prevents accumulation

        if epoch % (epochs / 5) == 0:
            loss = loss.item()
            print(f"loss: {loss:>7f}\t[{epoch:5d}/{epochs:>5d}]")

# Training the Model

In [ ]:
# Creating model, loss function and optimizer
model = NeuralNetwork(64)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# Training the model
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
train(X, y, model, loss_fn, optimizer, 2000, device)

# Testing the Model

In [ ]:
# Which integers to multiply
x1 = 9
x2 = 3

# Predicting outcome
model.eval()
model.to('cpu')
print(round(model(torch.tensor([x1, x2], dtype=torch.float32) / 10).item() * 100))